<a href="https://colab.research.google.com/github/Marce-Lopes/GDMC/blob/main/Analytics_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Q1a · Curva de Vintage — Inadimplência Acumulada

**Tabelas utilizadas:** `contrato` + `parcela`  
**Tabelas disponíveis mas fora do escopo do Q1:** `cliente`, `vendedor`, `loja`, `lead_campanha`, `produto_financeiro`  
→ Entram nas questões Q2 e Q3.

**Definições adotadas:**
- **Safra:** trimestre de originação do contrato (`data_contrato`)
- **Mês de vida:** delta em meses entre a `data_vencimento` da primeira parcela com `dias_atraso > 0` e a `data_contrato` — conforme Regra 1 do enunciado
- **Inadimplência acumulada:** % de contratos da safra que atingiram o critério até cada marco (M03, M06, M09, M12)
- **Formato de saída:** transposto — uma linha por safra, marcos como colunas — conforme Regra 2 do enunciado

**Lógica em 3 CTEs:**
```
contrato → safra trimestral
    ↓
parcela → primeira ocorrência de dias_atraso > 0 por contrato → mês de vida
    ↓
agregação por safra → % acumulado até M03 / M06 / M09 / M12
```

In [4]:
sql_q1a = """
WITH safra AS (
    SELECT
        id_contrato,
        data_contrato,
        STRFTIME('%Y', data_contrato) || '-Q' ||
            CAST(CEIL(CAST(STRFTIME('%m', data_contrato) AS REAL) / 3) AS INTEGER) AS safra
    FROM contrato
),
primeiro_atraso AS (
    SELECT
        id_contrato,
        MIN(data_vencimento) AS data_primeiro_atraso
    FROM parcela
    WHERE dias_atraso > 0
    GROUP BY id_contrato
),
mes_vida AS (
    SELECT
        s.safra,
        s.id_contrato,
        ROUND(
            (JULIANDAY(pa.data_primeiro_atraso) - JULIANDAY(s.data_contrato)) / 30.0
        ) AS mes_inadimplencia
    FROM safra s
    LEFT JOIN primeiro_atraso pa ON s.id_contrato = pa.id_contrato
),
acumulado AS (
    SELECT
        safra,
        COUNT(id_contrato) AS total_contratos,
        ROUND(100.0 * SUM(CASE WHEN mes_inadimplencia <= 3  THEN 1 ELSE 0 END) / COUNT(*), 1) AS M03,
        ROUND(100.0 * SUM(CASE WHEN mes_inadimplencia <= 6  THEN 1 ELSE 0 END) / COUNT(*), 1) AS M06,
        ROUND(100.0 * SUM(CASE WHEN mes_inadimplencia <= 9  THEN 1 ELSE 0 END) / COUNT(*), 1) AS M09,
        ROUND(100.0 * SUM(CASE WHEN mes_inadimplencia <= 12 THEN 1 ELSE 0 END) / COUNT(*), 1) AS M12
    FROM mes_vida
    GROUP BY safra
)
SELECT * FROM acumulado ORDER BY safra
"""

df_q1a = pd.read_sql_query(sql_q1a, conn)
# Substitua o display(df_q1a) por:
display(
    df_q1a.style
    .format(precision=1)
    .set_caption("Q1a · Inadimplência Acumulada por Safra (%)").hide(axis='index')
)

safra,total_contratos,M03,M06,M09,M12
2020-Q1,102,30.4,41.2,52.9,53.9
2020-Q2,94,33.0,48.9,54.3,58.5
2020-Q3,105,43.8,53.3,61.9,65.7
2020-Q4,101,20.8,37.6,50.5,52.5
2021-Q1,104,35.6,50.0,51.0,52.9
2021-Q2,109,33.9,46.8,52.3,57.8
2021-Q3,112,22.3,36.6,43.8,46.4
2021-Q4,110,30.9,46.4,51.8,57.3
2022-Q1,124,29.8,48.4,50.8,56.5
2022-Q2,102,26.5,40.2,49.0,53.9


---
## Q1b · Ranking de Safras por Risco

**Tabelas utilizadas:** `contrato` + `parcela` — extensão direta do Q1a, nenhum dado novo.

**Definições adotadas:**
- **Ano:** extraído da safra (ex: `2022-Q1` → `2022`)
- **Rank:** 1 = safra mais arriscada do ano — `RANK() OVER (PARTITION BY ano ORDER BY M12 DESC)` — conforme Regra 3
- **Desvio:** diferença em pontos percentuais entre o M12 da safra e a média do seu ano — `M12 - AVG(M12) OVER (PARTITION BY ano)`

**Lógica em 2 CTEs:**
```
Q1a completo (safra + M12)

    ↓

RANK() e AVG() OVER (PARTITION BY ano)

    ↓

uma linha por safra com rank e desvio vs média
```

In [5]:


sql_q1b = """
WITH safra AS (
    SELECT
        id_contrato,
        data_contrato,
        STRFTIME('%Y', data_contrato)                           AS ano,
        STRFTIME('%Y', data_contrato) || '-Q' ||
            CAST(CEIL(CAST(STRFTIME('%m', data_contrato)
            AS REAL) / 3) AS INTEGER)                          AS safra
    FROM contrato
),
primeiro_atraso AS (
    SELECT
        id_contrato,
        MIN(data_vencimento) AS data_primeiro_atraso
    FROM parcela
    WHERE dias_atraso > 0
    GROUP BY id_contrato
),
mes_vida AS (
    SELECT
        s.safra,
        s.ano,
        s.id_contrato,
        ROUND(
            (JULIANDAY(pa.data_primeiro_atraso) - JULIANDAY(s.data_contrato)) / 30.0
        ) AS mes_inadimplencia
    FROM safra s
    LEFT JOIN primeiro_atraso pa ON s.id_contrato = pa.id_contrato
),
m12_por_safra AS (
    SELECT
        safra,
        ano,
        COUNT(id_contrato)  AS total_contratos,
        ROUND(100.0 * SUM(CASE WHEN mes_inadimplencia <= 12 THEN 1 ELSE 0 END)
              / COUNT(id_contrato), 2) AS M12
    FROM mes_vida
    GROUP BY safra, ano
)
SELECT
    safra,
    ano,
    total_contratos,
    M12                                                         AS pct_inadimplente_m12,
    RANK() OVER (
        PARTITION BY ano
        ORDER BY M12 DESC
    )                                                           AS rank_risco_no_ano,
    ROUND(AVG(M12) OVER (PARTITION BY ano), 2)                 AS media_ano,
    ROUND(M12 - AVG(M12) OVER (PARTITION BY ano), 2)          AS desvio_vs_media_pp
FROM m12_por_safra
ORDER BY ano, rank_risco_no_ano
"""

df_q1b = pd.read_sql_query(sql_q1b, conn)

display(df_q1b.style
    .set_caption("Q1b · Ranking de Safras por Risco (M12)")
    .format({
        'pct_inadimplente_m12': '{:.2f}%',
        'media_ano':            '{:.2f}%',
        'desvio_vs_media_pp':   '{:+.2f}pp',
    })

)

,safra,ano,total_contratos,pct_inadimplente_m12,rank_risco_no_ano,media_ano,desvio_vs_media_pp
0,2020-Q3,2020,105,65.71%,1,57.66%,+8.06pp
1,2020-Q2,2020,94,58.51%,2,57.66%,+0.86pp
2,2020-Q1,2020,102,53.92%,3,57.66%,-3.73pp
3,2020-Q4,2020,101,52.48%,4,57.66%,-5.17pp
4,2021-Q2,2021,109,57.80%,1,53.60%,+4.20pp
5,2021-Q4,2021,110,57.27%,2,53.60%,+3.68pp
6,2021-Q1,2021,104,52.88%,3,53.60%,-0.71pp
7,2021-Q3,2021,112,46.43%,4,53.60%,-7.17pp
8,2022-Q1,2022,124,56.45%,1,54.08%,+2.37pp
9,2022-Q3,2022,90,55.56%,2,54.08%,+1.48pp




---



---
### Q1a · Interpretação: Curva de Vintage

- Todas as safras apresentam inadimplência acumulada entre **50% e 66% no M12**
- **2020-Q3** é a safra de maior risco: 43.8% de inadimplência já no M03,
  encerrando em 65.7% no M12
- **2021-Q3** é a safra de menor risco: única abaixo de 50% no M12 (46.4%)
- Em todas as safras, a maior aceleração ocorre entre M03 e M06 — os primeiros
  6 meses de vida do contrato concentram a maior parte dos eventos de inadimplência

### Q1b · Interpretação: Ranking por Risco

- Em 2020, **2020-Q3** apresenta o maior desvio positivo: +8.06pp acima da média do ano (57.66%)
- Em 2021, **2021-Q2** (+4.20pp) e **2021-Q4** (+3.68pp) ficam acima da média (53.60%)
- Em 2022, as safras ficam próximas da média (54.08%) — menor dispersão do período analisado
- Em 2023, **2023-Q4** lidera o risco com +3.93pp acima da média (55.29%)
- Nenhuma safra em nenhum ano ficou abaixo de 46% no M12

---

### Nota Técnica - Camada DBT

| Model | Camada | Materialização | Motivo |
|---|---|---|---|
| stg_contrato | staging | `view` | Só tipagem de data e cálculo de safra — leve |
| stg_parcela | staging | `view` | Filtro simples — não justifica persistir |
| int_primeiro_atraso` | intermediate | `table` | Agrega 29k parcelas — caro recomputar |
| int_mes_vida | intermediate | `table` | Join pesado — mesma justificativa |
| fin_vintage_q1a | marts | `table` | Entrega final consumida pelo BI |
| fin_ranking_q1b | marts | `table` | Idem |

> Em volume maior, `int_primeiro_atraso` e `int_mes_vida` migrariam para `incremental`
> — reprocessando só contratos novos por `data_contrato`, sem refazer o histórico.

#### **Como ler os resultados (time financeiro, time risco)**

*Time de Risco — `fin_vintage_q1a`*
Cada linha representa uma safra trimestral. As colunas M03, M06, M09 e M12
mostram o percentual acumulado de contratos que entraram em inadimplência
até aquele marco de vida do contrato. Use para identificar safras com
deterioração precoce (M03 alto) e calibrar o PDD por coorte.

*Time Financeiro — `fin_ranking_q1b`*
Cada linha representa uma safra com seu percentual de inadimplência no M12,
posição de risco dentro do ano (rank_risco_no_ano) e distância da média anual
(desvio_vs_media_pp). Valores positivos indicam safras acima da média do ano;
valores negativos, abaixo. Use para priorizar quais safras demandam
provisão adicional no fechamento mensal.

### Camada dbt e Materialização

O modelo de vintage percorre três camadas:

- **Staging** — `stg_contrato` e `stg_parcela` ficam como `view`: tipagem e renomeação de colunas, sem lógica de negócio.
- **Intermediate** — `int_primeiro_atraso` e `int_mes_vida` viram `table`: agregam 29k parcelas e executam joins pesados, recomputar a cada execução seria custoso.
- **Marts** — `fin_vintage` é `table` incremental particionada por safra: novas safras são appendadas sem reprocessar o histórico.

### Exposição do DBT para os times de Risco e Financeiro

A tabela `fin_vintage` entrega uma linha por safra com os percentuais de inadimplência acumulada em M03, M06, M09 e M12 já calculados.

Para consulta, o time acessa diretamente via Databricks SQL — sem necessidade de conhecer as tabelas brutas ou a lógica de cálculo. Cada coluna está documentada no dbt docs com definição de negócio. Exemplo: *"M06 = percentual de contratos da safra com primeira parcela atrasada registrada até 6 meses após a assinatura do contrato."*

O modelo é atualizado diariamente com alerta automático em caso de falha, garantindo que os números estejam disponíveis com hora certa e rastreabilidade de execução.




---





---



---

## Q2 · Reativação de Cartão

**Descrição:** Com a qualidade das safras mapeada, o foco se volta para a base ativa de clientes.
Q2a identifica oportunidades de reativação via campanha; Q2b sinaliza concentração de risco
para o time de crédito. Ambas alimentam decisões de negócio distintas a partir do mesmo núcleo de dados.

---
### Q2a · Campanha de Reativação de Cartão

**Tabelas utilizadas:** `contrato` + `cliente` + `produto_financeiro` + `lead_campanha`  
**Tabelas disponíveis mas fora do escopo do Q2a:** `parcela`, `vendedor`, `loja`  
→ `parcela` entra no Q2b para cálculo de atraso e total em aberto.

**Definições adotadas:**
- **Domínio válido:** `@gmail.com` ou `@yahoo.com.br` — conforme Regra 4
- **Janela de originação:** contratos de cartão entre 2021 e 2023, entre 2 e 3 contratos — conforme Regra 5
- **Sem inadimplência:** nenhum contrato com `status = 'inadimplente'` em qualquer produto — conforme Regra 6
- **Sem engajamento recente:** nenhum `status_resposta` igual a `clicou` ou `contratou` nos últimos 6 meses — conforme Regra 7

**Lógica em 4 CTEs:**
```
cliente → filtro por domínio de e-mail
↓
contrato + produto_financeiro → filtro tipo cartao + janela 2021-2023 + count entre 2 e 3
↓
contrato → exclusão de clientes com qualquer contrato inadimplente
↓
lead_campanha → exclusão de clientes com clique ou conversão nos últimos 6 meses
↓
retorno: nome, email, estado, canal_aquisicao, qtd_contratos_cartao, valor_medio_contratado
```

**Conformidade com a LGPD antes da entrega da lista:**

A query aplica minimização de dados na origem — CPF, renda e score não são mostrados no output, expondo apenas o necessário para a campanha. Caso o contrário, dados sensiveis classificadas como pessoais (PI) devem ser anonimizados definitivamente a fim de tornar impossível de identificar o indivíduo da informação. Antes da entrega, a lista deve ser cruzada com a lista de preferências de comunicação para exclusão de clientes em opt-out, critério obrigatório independente dos filtros de campanha aplicados. A lista deve ser compartilhada exclusivamente em ambiente com RBAC (Databricks/S3). Ao fim da campanha, a lista deve ser descartada com registro auditável.

In [6]:
sql_q2a = """
-- ================================================================
-- Q2a · Campanha de Reativação de Cartão
-- Retorna clientes elegíveis para reativação com base em 4 critérios
-- simultâneos de inclusão e exclusão
-- ================================================================

WITH cartoes_por_cliente AS (
    -- Contratos de cartão originados entre 2021 e 2023
    -- HAVING aplicado na agregação — evita subquery de exclusão posterior
    SELECT
        c.id_cliente,
        COUNT(c.id_contrato)             AS qtd_contratos_cartao,
        ROUND(AVG(c.valor_contratado), 2) AS valor_medio_contratado
    FROM contrato c
    INNER JOIN produto_financeiro pf ON c.id_produto = pf.id_produto
    WHERE pf.tipo = 'cartao'
      AND STRFTIME('%Y', c.data_contrato) BETWEEN '2021' AND '2023'
    GROUP BY c.id_cliente
    HAVING COUNT(c.id_contrato) BETWEEN 2 AND 3
),

inadimplentes AS (
    -- Clientes com ao menos um contrato inadimplente em qualquer produto
    -- DISTINCT + EXISTS: evita explosão de linhas em clientes com múltiplos contratos
    SELECT DISTINCT id_cliente
    FROM contrato
    WHERE status = 'inadimplente'
),

engajados_recentes AS (
    -- Clientes com clique ou conversão nos últimos 6 meses
    -- Em produção: substituir DATE('now') por parâmetro de execução
    -- para garantir idempotência em reprocessamentos
    SELECT DISTINCT id_cliente
    FROM lead_campanha
    WHERE status_resposta IN ('clicou', 'contratou')
      AND data_disparo >= DATE('now', '-6 months')
)

SELECT
    cl.nome,
    cl.email,
    cl.estado,
    cl.canal_aquisicao,
    cc.qtd_contratos_cartao,
    cc.valor_medio_contratado
FROM cliente cl
INNER JOIN cartoes_por_cliente cc ON cl.id_cliente = cc.id_cliente
WHERE (lower(cl.email) LIKE '%@gmail.com' OR lower(cl.email) LIKE '%@yahoo.com.br')
  AND NOT EXISTS (SELECT 1 FROM inadimplentes    i WHERE i.id_cliente = cl.id_cliente)
  AND NOT EXISTS (SELECT 1 FROM engajados_recentes e WHERE e.id_cliente = cl.id_cliente)
ORDER BY cl.nome
"""

df_q2a = pd.read_sql_query(sql_q2a, conn)

display(
    df_q2a.style
    .format({'valor_medio_contratado': 'R$ {:,.2f}'})
    .set_caption("Q2a · Clientes Elegíveis para Reativação de Cartão")
    .hide(axis='index')
)

nome,email,estado,canal_aquisicao,qtd_contratos_cartao,valor_medio_contratado
Aline Barbosa Almeida,aline892@gmail.com,AM,loja_fisica,3,"R$ 3,255.12"
Aline Moreira Silva,aline670@yahoo.com.br,PA,loja_fisica,3,"R$ 3,563.44"
Amanda Almeida Fernandes,cliente5@yahoo.com.br,RO,televendas,2,"R$ 3,459.70"
Ana Alves Vieira,ana434@gmail.com,GO,loja_fisica,3,"R$ 4,370.90"
Ana Araújo Martins,ana580@gmail.com,DF,site,2,"R$ 4,632.37"
Ana Barbosa Fernandes,ana625@gmail.com,RO,loja_fisica,3,"R$ 9,224.31"
Ana Ribeiro Gomes,ana220@gmail.com,MT,site,2,R$ 503.72
André Alves Alves,andre434@gmail.com,DF,loja_fisica,2,"R$ 2,602.50"
André Lima Gomes,andre959@yahoo.com.br,RO,site,2,"R$ 1,423.16"
Beatriz Silva Oliveira,beatriz691@gmail.com,AM,loja_fisica,2,"R$ 3,650.10"


---
### Q2b · Identificação de Clientes de Alto Risco

**Tabelas utilizadas:** `contrato` + `parcela` + `cliente`  
**Tabelas disponíveis mas fora do escopo do Q2b:** `produto_financeiro`, `lead_campanha`, `vendedor`, `loja`  
→ Q2b foca em comportamento de pagamento e perfil de renda, independente de produto ou campanha.

**Definições adotadas:**
- **Contratos ativos:** `status = 'ativo'`, dois ou mais por cliente — conforme Regra 8
- **Atraso recente:** pelo menos uma parcela com `dias_atraso > 30` nos últimos 90 dias — conforme Regra 8
- **Renda baixa:** `renda_declarada` abaixo da mediana da base — conforme Regra 9
- **Total em aberto:** `SUM(valor_parcela - valor_pago)` nas parcelas com `status IN ('atrasada', 'em_aberto')` — conforme Regra 11

**Lógica em 3 CTEs:**
```
contrato → clientes com 2+ contratos ativos
↓
parcela → clientes com dias_atraso > 30 nos últimos 90 dias
↓
cliente → filtro renda_declarada abaixo da mediana + cálculo do total em aberto
↓
retorno: nome, score_interno, qtd_contratos_ativos, total_em_aberto
```

In [7]:
sql_q2b = """
-- ================================================================
-- Q2b · Identificação de Clientes de Alto Risco
-- Retorna clientes que acumulam simultaneamente: 2+ contratos ativos,
-- atraso > 30 dias nos últimos 90 dias e renda abaixo da mediana
-- ================================================================

WITH mediana_renda AS (
    -- Mediana calculada uma única vez e reutilizada no filtro final
    -- LIMIT/OFFSET: método portável em SQLite — evita funções de percentil
    -- não disponíveis nativamente
    SELECT AVG(renda_declarada) AS mediana
    FROM (
        SELECT renda_declarada
        FROM cliente
        ORDER BY renda_declarada
        LIMIT 2 - (SELECT COUNT(*) FROM cliente) % 2
        OFFSET (SELECT (COUNT(*) - 1) / 2 FROM cliente)
    )
),

contratos_ativos AS (
    -- Clientes com 2 ou mais contratos ativos — conforme Regra 8
    SELECT
        id_cliente,
        COUNT(id_contrato) AS qtd_contratos_ativos
    FROM contrato
    WHERE status = 'ativo'
    GROUP BY id_cliente
    HAVING COUNT(id_contrato) >= 2
),

atraso_recente AS (
    -- Clientes com ao menos uma parcela com dias_atraso > 30
    -- nos últimos 90 dias — conforme Regra 8
    -- DISTINCT: um cliente pode ter múltiplas parcelas em atraso;
    -- só precisamos confirmar a existência
    SELECT DISTINCT c.id_cliente
    FROM parcela p
    INNER JOIN contrato c ON p.id_contrato = c.id_contrato
    WHERE p.dias_atraso > 30
    AND p.data_vencimento >= DATE(
            (SELECT MAX(data_vencimento) FROM parcela), '-90 days'
          )
),

total_em_aberto AS (
    -- Total em aberto por cliente: soma de valor_parcela - valor_pago
    -- restrito a parcelas atrasadas ou em aberto — conforme Regra 11
    SELECT
        c.id_cliente,
        ROUND(SUM(p.valor_parcela - p.valor_pago), 2) AS total_em_aberto
    FROM parcela p
    INNER JOIN contrato c ON p.id_contrato = c.id_contrato
    WHERE p.status IN ('atrasada', 'em_aberto')
    GROUP BY c.id_cliente
)

SELECT
    cl.nome,
    cl.score_interno,
    ca.qtd_contratos_ativos,
    COALESCE(ta.total_em_aberto, 0) AS total_em_aberto
FROM cliente cl
INNER JOIN contratos_ativos  ca ON cl.id_cliente = ca.id_cliente
INNER JOIN atraso_recente    ar ON cl.id_cliente = ar.id_cliente
INNER JOIN total_em_aberto   ta ON cl.id_cliente = ta.id_cliente
WHERE cl.renda_declarada < (SELECT mediana FROM mediana_renda)
ORDER BY ta.total_em_aberto DESC
"""

df_q2b = pd.read_sql_query(sql_q2b, conn)

display(
    df_q2b.style
    .format({'total_em_aberto': 'R$ {:,.2f}'})
    .set_caption("Q2b · Clientes de Alto Risco")
    .hide(axis='index')
)

nome,score_interno,qtd_contratos_ativos,total_em_aberto
Daniela Lopes Fernandes,811,2,"R$ 128,143.53"
Mariana Ferreira Ferreira,415,4,"R$ 93,253.02"
Paulo Gomes Almeida,638,4,"R$ 91,503.50"
Camila Barbosa Souza,295,7,"R$ 81,112.48"
Vanessa Martins Vieira,910,7,"R$ 76,554.87"
Beatriz Martins Ribeiro,944,4,"R$ 68,792.25"
André Barbosa Costa,279,4,"R$ 68,402.14"
Henrique Costa Pereira,723,4,"R$ 67,182.96"
Patrícia Santos Barbosa,361,4,"R$ 60,008.43"
Larissa Almeida Souza,812,3,"R$ 57,948.05"


**Assumptions adotados — Q2a e Q2b:**

1. **Janela de 6 meses em Q2a** — `DATE('now', '-6 months')` ancorado no relógio do servidor. Em produção, substituir por parâmetro de execução para garantir idempotência em reprocessamentos.
2. **Janela de 90 dias em Q2b** — ancorada em `MAX(data_vencimento)` da base em vez de `now()`, evitando retorno vazio em dados históricos.
3. **`status = 'inadimplente'` no contrato** — assume que o status do contrato é atualizado em sincronia com as parcelas. Um contrato com parcelas atrasadas mas `status = 'ativo'` passaria pelo filtro.
4. **Mediana calculada sobre toda a base de clientes** — incluindo clientes sem produto ativo. Mediana é populacional, não restrita a clientes com contrato vigente.
5. **`data_vencimento` como proxy dos 90 dias** — consistente com a decisão do Q1a de não usar `data_pagamento` dado o volume de nulos.
6. **Opt-out não modelado** — assumido como existente no CRM mas fora do escopo do banco disponível. Documentado também na seção LGPD.

---
---
### Q2a · Interpretação: Campanha de Reativação

- **50 clientes** atendem simultaneamente aos 4 critérios de elegibilidade
- **`loja_fisica`** é o canal dominante — presente em 24 dos 50 clientes — seguido de `site` (15), `app` (7) e `televendas` (4)
- **SP, RO e BA** são os estados com maior concentração de elegíveis — relevante para priorização regional da campanha
- O valor médio contratado varia de **R$ 503 a R$ 9.985** — segmentação por faixa de valor pode aumentar a conversão
- Clientes com **3 contratos de cartão** tendem a ter ticket médio mais alto — candidatos prioritários para oferta de upgrade de limite

### Q2b · Interpretação: Clientes de Alto Risco

- **46 clientes** acumulam simultaneamente os três fatores de risco
- **Daniela Lopes Fernandes** lidera o total em aberto com **BRL 128.143** — score 811, o que sinaliza deterioração recente não capturada pelo score interno
- **5 clientes** possuem score acima de 900 com total em aberto significativo — possível defasagem do score interno em relação ao comportamento real de pagamento
- Clientes com **7 contratos ativos** (Camila Barbosa Souza e Vanessa Martins Vieira) representam concentração de risco individual elevada — candidatos a revisão de limite imediata
- O total consolidado em aberto da carteira de risco supera **R$ 1,1 milhão** — dado relevante para o time de provisionamento calibrar o PDD

**Garantia de qualidade para o time de Data Science**

Três dimensões de qualidade cobertas via testes dbt: completude — `not_null` em
`id_cliente`, `renda_declarada` e `dias_atraso`, garantindo que nenhum campo crítico
para o modelo chegue vazio; unicidade — `unique` em `id_cliente` na saída, evitando
que um cliente seja contado duas vezes no treino; distribuição — teste customizado que
valida se a mediana de `renda_declarada` está dentro de um intervalo histórico esperado,
impedindo que uma carga corrompida altere silenciosamente o critério de corte.
SLA diário com alerta no CloudWatch em caso de falha — o modelo nunca treina com
dado desatualizado. Cada campo documentado no dbt docs com definição de negócio e
os assumptions desta query registrados como notas do modelo.

---
## Q3 · Enriquecimento de Análises e Feature Store

**Descrição:** O Q3 opera em dois momentos distintos. Q3a é execução analítica —
enriquecimento da base interna com dados externos do Serasa para identificar
oportunidades de upgrade e concentração de propensão a crédito. Q3b é arquitetura —
como esses dados enriquecidos chegam ao time de Data Science como feature store
no Databricks, com garantias de qualidade, rastreabilidade e point-in-time correctness.

---
### Q3a · Enriquecimento e Análise

**Tabelas utilizadas:** `cliente` + `contrato` + `loja` (FinVarejo.db) + `serasa_scores.csv`
**Chave de join:** `cpf`

**Definições adotadas:**
- **Delta de score:** `score_mercado - score_interno` — conforme Regra 12
- **Top 10:** clientes com maior delta positivo — candidatos a upgrade de limite
- **Top 3 regiões:** `propensao_credito = 'Alta'` agrupado por `regiao` da tabela `loja`,
  acessada via `contrato → loja` — conforme Regra 14
- **Gráfico:** boxplot de `score_mercado` por `canal_aquisicao` — conforme Regra 15

**Lógica:**
```
serasa_scores.csv → join com cliente via CPF
    ↓
delta = score_mercado - score_interno → Top 10
    ↓
cliente → contrato → loja → regiao
    ↓
propensao_credito = 'Alta' → Top 3 regiões
    ↓
boxplot score_mercado por canal_aquisicao
```

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# ================================================================
# Q3a · Enriquecimento e Análise — Serasa + FinVarejo.db
# Assumption: serasa_scores.csv não está disponível no ambiente
# de execução. O código assume a existência do arquivo com as
# colunas cpf, score_mercado, faixa_renda_estimada,
# propensao_credito e data_referencia — conforme enunciado.
# ================================================================

conn = sqlite3.connect('FinVarejo.db')

# ----------------------------------------------------------------
# Leitura e join base
# CPF como chave — LOWER() em ambos os lados por consistência
# com a decisão de normalização adotada no Q2a
# ----------------------------------------------------------------
serasa = pd.read_csv('serasa_scores.csv')
serasa['cpf'] = serasa['cpf'].str.strip().str.lower()

cliente = pd.read_sql_query('SELECT * FROM cliente', conn)
cliente['cpf'] = cliente['cpf'].str.strip().str.lower()

# Inner join — só clientes presentes em ambas as bases
df = cliente.merge(serasa, on='cpf', how='inner')

# ----------------------------------------------------------------
# Top 10 · Maior delta positivo entre score_mercado e score_interno
# Regras 12 e 13
# ----------------------------------------------------------------
df['delta_score'] = df['score_mercado'] - df['score_interno']

top10 = (
    df[df['delta_score'] > 0]
    .nlargest(10, 'delta_score')
    [['nome', 'score_mercado', 'score_interno', 'delta_score', 'canal_aquisicao']]
    .reset_index(drop=True)
)

display(
    top10.style
    .format({'delta_score': '+{:.0f}', 'score_mercado': '{:.0f}', 'score_interno': '{:.0f}'})
    .set_caption("Q3a · Top 10 Clientes — Candidatos a Upgrade de Limite")
    .hide(axis='index')
)

# ----------------------------------------------------------------
# Top 3 regiões · Concentração de propensao_credito = 'Alta'
# Requer join cliente → contrato → loja para obter regiao
# Regra 14
# ----------------------------------------------------------------
contrato = pd.read_sql_query(
    'SELECT DISTINCT id_cliente, id_loja FROM contrato', conn
)
loja = pd.read_sql_query(
    'SELECT id_loja, regiao FROM loja', conn
)

# Join em cadeia: df (com Serasa) → contrato → loja
df_regiao = (
    df[df['propensao_credito'] == 'Alta']
    .merge(contrato, on='id_cliente', how='left')
    .merge(loja, on='id_loja', how='left')
)

top3_regioes = (
    df_regiao
    .groupby('regiao')['id_cliente']
    .nunique()                      # nunique: evita contar o mesmo cliente
    .reset_index()                  # duas vezes se tiver múltiplos contratos
    .rename(columns={'id_cliente': 'qtd_clientes'})
    .nlargest(3, 'qtd_clientes')
    .reset_index(drop=True)
)

display(
    top3_regioes.style
    .set_caption("Q3a · Top 3 Regiões — Concentração de Alta Propensão a Crédito")
    .hide(axis='index')
)

# ----------------------------------------------------------------
# Gráfico · Distribuição de score_mercado por canal_aquisicao
# Paleta de cores da marca — consistente com o grafo de modelo
# de dados utilizado no documento
# Regra 15
# ----------------------------------------------------------------
paleta_marca = {
    'loja_fisica': '#F4A261',
    'site':        '#E76F51',
    'app':         '#B5EAD7',
    'televendas':  '#A7C7E7',
}

fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='canal_aquisicao',
    y='score_mercado',
    palette=paleta_marca,
    width=0.5,
    linewidth=1.2,
    ax=ax
)

ax.set_title('Distribuição de Score Mercado por Canal de Aquisição', fontsize=13)
ax.set_xlabel('Canal de Aquisição', fontsize=11)
ax.set_ylabel('Score Mercado', fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

---
### Q3b · Arquitetura de Feature Store

**Descrição:** Os dados enriquecidos do Q3a precisam ser entregues ao time de Data
Science como feature store no Databricks — com garantias de qualidade, rastreabilidade
e point-in-time correctness. A arquitetura adotada segue a camada medallion no S3 +
Delta Lake, com o AWS Glue Data Catalog como metastore e o Databricks como motor
central de transformação e consumo.


---
### Q3b · Arquitetura de Feature Store

Decisão central: medallion no S3 +
Delta Lake, Databricks como motor, Glue só como metastore.

O Glue como motor de transformação adicionaria um componente a mais sem ganho real, o Delta Lake nativo no Databricks resolve idempotência, time travel e schema evolution no mesmo ambiente.

#### Camadas Bronze / Silver / Gold

Bronze recebe o CSV bruto particionado por `ingestion_date` — fonte de verdade imutável. Silver limpa, deduplica e faz o join com `cliente` via CPF e cada feature recebe `event_timestamp` e `created_timestamp`. Sem isso, o modelo pode usar informação do futuro sem perceber. Com isso, Gold entrega features agregadas prontas para consumo pelo cientista de dados.

#### Glue Data Catalog

├── db_bronze  →  serasa_raw            (particionado por ingestion_date)

├── db_silver  →  serasa_enriched       (particionado por event_date)

└── db_gold    →  feature_score_mercado · feature_propensao_credito

Uma database por camada: acesso granular via IAM + Lake Formation, lineage rastreável para auditoria BACEN e schema evolution independente sem risco de quebrar o contrato do Gold.

#### Point-in-time Correctness

Uma view calcula no momento da consulta — sempre com o estado atual. O Delta Lake time travel garante snapshots históricos reproduzíveis para retraining e auditoria, assim evita que o modelo performe bem no treino e mal em produção

#### SLA

| Etapa | SLA | Monitoramento |
|---|---|---|
| Bronze | até 02h UTC | CloudWatch |
| Silver | até 04h UTC | Lakehouse Monitoring |
| Gold | até 06h UTC | Alerta ao time de DS |

#### Feature Store vs View

**Feature store** para treino e revalidação de modelos — point-in-time correctness,
contrato de schema versionado e monitoramento de drift. Em ambientes com alta criticidade é necessário sempre ter documentação e histórico de registros e atividades que a feature store entrega melhor que a view

**View** para exploração ad hoc e prototipação — sem snapshots históricos. Views para essa situação em especial são mais apropriadas para um entrega pontual e rápida, também considerando a sensibilidade da informação.

---
## Q4 · Arquitetura de Pipeline — Ingestão Batch e Tempo Real

O CRM da FinVarejo disponibiliza dois mecanismos de extração com restrições relevantes:
Export Raw User Data exporta até 2TB via link S3 pré-assinado que expira em 24h,
disponível 1x/dia. O Webhook envia eventos em tempo real para um endpoint configurado.
A arquitetura pensada precisa endereçar os dois sem criar dependência entre eles.

---
### Q4a · Pipeline de Ingestão Batch

#### Decisão de stack

O Databricks Jobs vira motor de transformação — o Glue adicionaria um
componente a mais na stack sem ganho real, dado que o Delta Lake é nativo no Databricks. Ele também resolve idempotência, time travel e monitoramento no mesmo ambiente. O MWAA orquestra acionando um job Databricks ao invés de um crawler Glue.

#### Agendamento e idempotência

O DAG dispara às 02h UTC com timeout de 4h — margem suficiente para o SLA de 06h com espaço para retry. A primeira tarefa do DAG é persistir o arquivo bruto no S3 Landing imediatamente após o download. Essa é uma das principais tarefas dado o fato que o link expira em 24h e o arquivo
tem 2TB — se o processamento falhar após o download, o arquivo já está seguro e o retry não depende do link.

Idempotência em duas camadas:

- **S3 sensor** verifica `_SUCCESS` no path da `ingestion_date` antes de processar. Se existir, a task downstream é skipada via `ShortCircuitOperator`
- **Delta MERGE** no Silver garante deduplicação mesmo se o job rodar duas vezes no mesmo dia. Para garantir o sucesso da tarefa, retry com backoff exponencial — 3 tentativas — e alarme no CloudWatch após falha final.

#### Formato, particionamento e compressão

| Decisão | Escolha | Motivo |
|---|---|---|
| Formato | Parquet + Delta Lake | Columnar, compressão nativa, time travel |
| Compressão | Snappy | Equilíbrio velocidade/tamanho |
| Particionamento Bronze | `ingestion_date` | Isolamento por carga diária |
| Particionamento Silver/Gold | `event_date` | Alinhado com janela de treino |

#### SLA e monitoramento

| Etapa | SLA | Monitoramento |
|---|---|---|
| S3 Landing | até 03h UTC | CloudWatch — alarme se ausente |
| Bronze | até 04h UTC | CloudWatch |
| Silver/Gold | até 06h UTC | Databricks Lakehouse Monitoring |

---
### Q4b · Integração Real-time e Decisão Arquitetural

#### Separação por criticidade

Definição por dois streams com responsabilidades distintas:

| Stream | Eventos | Consumidor |
|---|---|---|
| `stream-critico` | Fraude, chargeback, bloqueio | Lambda → DynamoDB → alerta imediato |
| `stream-padrao` | Compras, pagamentos, SMS, push | Kinesis Firehose → S3 → Bronze Delta |

O `stream-critico` bypassa o pipeline batch com latência de segundos, sem esperar a
janela UTC. O `stream-padrao` converge com o batch diário no Bronze.

O `stream-critico` usa Kinesis Data Streams — necessário para múltiplos consumidores
e controle de replay. O `stream-padrao` usa Kinesis Firehose pagando por GB ingerido,
sem gerenciar shards, mais simples para o fluxo que vai direto ao S3.

#### Deduplicação no Delta Lake

Nessa arquitetura, existe a possibilidade de um evento chegar pelo Webhook e pelo Export batch no mesmo dia. Nesses casos, a deduplicação
acontece no Silver via MERGE com `event_id` como chave natural:

```sql
MERGE INTO silver.crm_events AS target
USING stream_bronze AS source
ON target.event_id = source.event_id
AND target.event_date = source.event_date
WHEN NOT MATCHED THEN INSERT *
```

#### Controle de custo — Shard Hours

Duas métricas no CloudWatch definem se o dimensionamento está correto:
`WriteProvisionedThroughputExceeded` indica subprovisionamento — shards insuficientes para o volume. `GetRecords.IteratorAgeMilliseconds` alto indica que os consumidores não estão acompanhando a entrada — pode ser shard ou compute.

O Kinesis suporta auto scaling via Application Auto Scaling, mas operações de
split e merge têm custo e latência. Para o `stream-critico`, o dimensionamento conservador e revisão periódica por sazonalidade é mais seguro do que depender de scaling automático em ambientes controlados.
Para o `stream-padrao`, a decisão mais economica seria utilizar o kinesis firehose — cobra por GB ingerido, sem gerenciamento de shards, suficiente para consumidor único direto ao S3.

#### Vale manter dois pipelines?

Sim, pois resolvem problemas diferentes. O batch funciona para o dia a dia da operação, com o Export garantindo todos os eventos do dia, incluindo os que o Webhook pode ter perdido por falha de entrega. O real-time garante latência devido a criticidade de determinados eventos que não podem esperar pelo processamento em lote.

Depender só do Webhook cria lacunas na trilha de auditoria — o BACEN pode questionar o registro de uma transação e você não tem resposta. Depender só do batch significa descobrir uma fraude 6h depois, tempo suficiente para o dano se propagar.

O custo incremental do Kinesis (~$150/mês) é facilmente justificável frente ao risco regulatório e de fraude. Manter os dois é um mecanismo de segurança para a operação.


#### ADR — Decisão Arquitetural: Batch + Real-time

**Contexto:** O CRM exporta até 2TB/dia via link S3 pré-assinado com expiração de 24h e disponibilidade de 1 requisição/dia. Paralelamente, disponibiliza um Webhook de eventos em tempo real. As restrições do Export e a necessidade de detecção imediata de fraude tornam inviável resolver os dois casos de uso com um único pipeline. É uma situação em que optando por um dos dois caminhos ficaremos descobertos na outra ponta.

**Decisão:** Manter dois pipelines com responsabilidades distintas — batch para
completude e rastreabilidade, real-time para latência em eventos críticos.

**Trade-offs explícitos:**
- O dado de eventos recorrentes trafega duas vezes — Webhook e Export. O custo
  incremental (~$150/mês no Kinesis) é aceito; a deduplicação no Silver via MERGE
  garante que o dado final não duplica.
- Dois pipelines aumentam a superfície operacional. A mitigação é o MWAA centralizando
  a orquestração e o Lakehouse Monitoring cobrindo ambos.
- Depender só do Webhook cria lacunas na trilha de auditoria — risco regulatório direto
  com o BACEN. Depender só do batch significa detectar fraude com até 6h de atraso.

**Para stakeholders não-técnicos:** um pipeline resolve volume e auditoria, o outro resolve urgência. Abrir mão de qualquer um é risco — regulatório ou de fraude. O custo dos dois juntos é menor do que um incidente.